In [1]:
import os
import numpy as np
from tqdm import tqdm

In [2]:
BASE_DATASET_PATH = '../../../datasets/KITTI/dataset'
train_lbl_path = f'{BASE_DATASET_PATH}/train/labels/'

In [6]:
RESIZE_TO = 64

In [10]:

def load_kitti_boxes(label_path):

    boxes = []

    label_files = os.listdir(label_path)

    for file in tqdm(label_files):

        path = os.path.join(label_path, file)

        with open(path) as f:
            lines = f.readlines()

        for line in lines:
            cls, x, y, w, h = map(float, line.split())

            boxes.append([w, h])   # YOLO normalized

    return np.array(boxes)

In [11]:
def iou(box, clusters):

    x = np.minimum(clusters[:, 0], box[0])
    y = np.minimum(clusters[:, 1], box[1])

    intersection = x * y

    box_area = box[0] * box[1]
    cluster_area = clusters[:, 0] * clusters[:, 1]

    union = box_area + cluster_area - intersection

    return intersection / union

In [12]:
def kmeans(boxes, k=6, iters=100):

    indices = np.random.choice(len(boxes), k)
    clusters = boxes[indices]

    for _ in range(iters):

        distances = []

        for box in boxes:
            distances.append(1 - iou(box, clusters))

        distances = np.array(distances)

        nearest = np.argmin(distances, axis=1)

        new_clusters = []

        for i in range(k):
            new_clusters.append(
                np.median(boxes[nearest == i], axis=0)
            )

        new_clusters = np.array(new_clusters)

        if np.all(clusters == new_clusters):
            break

        clusters = new_clusters

    return clusters

In [17]:
boxes = load_kitti_boxes(train_lbl_path)

anchors = kmeans(boxes, k=6)

anchors = anchors[np.argsort(anchors[:,0] * anchors[:,1])]

print(anchors)

100%|████████████████████████████████████| 5981/5981 [00:00<00:00, 33317.93it/s]


[[0.0223591  0.06653333]
 [0.04203431 0.09376   ]
 [0.02941176 0.21881081]
 [0.07371981 0.14074667]
 [0.11732287 0.25908   ]
 [0.22706924 0.46122667]]


In [14]:
boxes = load_kitti_boxes(train_lbl_path)

anchors = kmeans(boxes, k=6)

# anchor area hesapla
areas = anchors[:,0] * anchors[:,1]

# median alan
median_area = np.median(areas)

# scale ayırımı
small_scale_anchors = anchors[areas < median_area]
large_scale_anchors = anchors[areas >= median_area]

#anchors = anchors[np.argsort(anchors[:,0] * anchors[:,1])]

small_scale_anchors = small_scale_anchors[
    np.argsort(small_scale_anchors[:,0] * small_scale_anchors[:,1])
]

large_scale_anchors = large_scale_anchors[
    np.argsort(large_scale_anchors[:,0] * large_scale_anchors[:,1])
]
ANCHORS = [
    large_scale_anchors.tolist(),   # S = 20
    small_scale_anchors.tolist(),   # S = 40
]

print(ANCHORS)

100%|████████████████████████████████████| 5981/5981 [00:00<00:00, 35967.90it/s]


[[[0.07298711755233493, 0.13976], [0.11818840579710142, 0.2549866666666667], [0.22644927536231885, 0.46143999999999996]], [[0.022181964573268976, 0.06691489361702134], [0.041674836601307226, 0.09325333333333333], [0.03084482271637788, 0.22739650623885915]]]


In [15]:
boxes = load_kitti_boxes(train_lbl_path)

anchors = kmeans(boxes, k=6)

aspect_ratio = anchors[:,0] / anchors[:,1]
areas = anchors[:,0] * anchors[:,1]

small_scale = anchors[
    (areas < np.percentile(areas, 50)) &
    (aspect_ratio < 1.2)
]

large_scale = anchors[
    (areas >= np.percentile(areas, 50)) |
    (aspect_ratio >= 1.2)
]

print(anchors)

100%|████████████████████████████████████| 5981/5981 [00:00<00:00, 34356.19it/s]


[[0.02941176 0.21881874]
 [0.11826892 0.26184   ]
 [0.02252818 0.06677333]
 [0.04247987 0.09442667]
 [0.07458132 0.14191892]
 [0.22852657 0.46226667]]


In [16]:
import numpy as np
import os
from tqdm import tqdm
import cv2
import glob


# -------------------------
# 1) LOAD BOXES (YOLO format)
# -------------------------
def load_kitti_boxes(label_path):
    boxes = []

    label_files = os.listdir(label_path)

    for file in tqdm(label_files):
        path = os.path.join(label_path, file)

        with open(path) as f:
            lines = f.readlines()

        for line in lines:
            split_line = line.strip().split()

            cls = split_line[0]

            # ignore DontCare (KITTI)
            if cls == "DontCare":
                continue

            x = float(split_line[1])
            y = float(split_line[2])
            w = float(split_line[3])
            h = float(split_line[4])

            boxes.append([w, h])

    return np.array(boxes)


# -------------------------
# 2) IOU FUNCTION
# -------------------------
def iou(box, clusters):
    x = np.minimum(clusters[:, 0], box[0])
    y = np.minimum(clusters[:, 1], box[1])

    intersection = x * y

    box_area = box[0] * box[1]
    cluster_area = clusters[:, 0] * clusters[:, 1]

    union = box_area + cluster_area - intersection

    return intersection / union


# -------------------------
# 3) KMEANS (YOLO STYLE)
# -------------------------
def kmeans(boxes, k=6, iters=100):
    indices = np.random.choice(len(boxes), k)
    clusters = boxes[indices]

    for _ in range(iters):

        distances = []

        for box in boxes:
            distances.append(1 - iou(box, clusters))

        distances = np.array(distances)

        nearest = np.argmin(distances, axis=1)

        new_clusters = []

        for i in range(k):
            cluster_boxes = boxes[nearest == i]

            if len(cluster_boxes) == 0:
                new_clusters.append(clusters[i])
            else:
                new_clusters.append(np.median(cluster_boxes, axis=0))

        new_clusters = np.array(new_clusters)

        if np.allclose(clusters, new_clusters):
            break

        clusters = new_clusters

    return clusters


# -------------------------
# 4) MAIN PIPELINE
# -------------------------
boxes = load_kitti_boxes(train_lbl_path)

anchors = kmeans(boxes, k=6)

# -------------------------
# 5) SPLIT (IMPORTANT FIX)
# -------------------------
areas = anchors[:, 0] * anchors[:, 1]
threshold = np.percentile(areas, 50)

small_scale = anchors[areas < threshold]
large_scale = anchors[areas >= threshold]

# -------------------------
# 6) SORT (for stability)
# -------------------------
small_scale = small_scale[np.argsort(small_scale[:, 0] * small_scale[:, 1])]
large_scale = large_scale[np.argsort(large_scale[:, 0] * large_scale[:, 1])]

# -------------------------
# 7) FINAL FORMAT
# -------------------------
ANCHORS = [
    large_scale.tolist(),  # S = 20 (coarse)
    small_scale.tolist()   # S = 40 (fine)
]

print("=== LARGE SCALE (S=20) ===")
print(large_scale)
torchvision.ops.batched_nms
print("\n=== SMALL SCALE (S=40) ===")
print(small_scale)

print("\n=== FINAL ANCHORS ===")
print(ANCHORS)

100%|████████████████████████████████████| 5981/5981 [00:00<00:00, 22540.94it/s]


=== LARGE SCALE (S=20) ===
[[0.10326087 0.18736   ]
 [0.05389718 0.37977298]
 [0.20632448 0.43978667]]

=== SMALL SCALE (S=40) ===
[[0.02609489 0.06818667]
 [0.02043301 0.17194927]
 [0.05130435 0.10877333]]

=== FINAL ANCHORS ===
[[[0.10326086956521739, 0.18735999999999997], [0.053897177228379306, 0.37977297682709443], [0.20632447665056355, 0.43978666666666666]], [[0.026094889646679914, 0.06818666666666673], [0.020433006535947717, 0.17194926916221037], [0.05130434782608698, 0.10877333333333339]]]


In [9]:
anchors_norm = np.array([
    [10,13], [16,30], [33,23],
    [30,61], [62,45], [59,119],
    [116,90], [156,198], [373,326]
], dtype=np.float32) / 640

In [10]:
anchors_norm

array([[0.015625 , 0.0203125],
       [0.025    , 0.046875 ],
       [0.0515625, 0.0359375],
       [0.046875 , 0.0953125],
       [0.096875 , 0.0703125],
       [0.0921875, 0.1859375],
       [0.18125  , 0.140625 ],
       [0.24375  , 0.309375 ],
       [0.5828125, 0.509375 ]], dtype=float32)

In [11]:
anchors = anchors_norm[np.argsort(anchors_norm[:,0] * anchors_norm[:,1])]

small_scale = anchors[:3]
large_scale = anchors[3:6]

In [12]:
small_scale

array([[0.015625 , 0.0203125],
       [0.025    , 0.046875 ],
       [0.0515625, 0.0359375]], dtype=float32)

In [13]:
large_scale

array([[0.046875 , 0.0953125],
       [0.096875 , 0.0703125],
       [0.0921875, 0.1859375]], dtype=float32)